# Cross-Sectional Stock Return Prediction

This notebook runs the attention experiment from `run_project_mhattn.py`.
Each model uses its own hyperparameters so the notebook matches the script-based runs more closely.


In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import stock_return_project4 as srp
import run_project_mhattn as mh

srp = importlib.reload(srp)
mh = importlib.reload(mh)
print(srp.__file__)
print(mh.__file__)


/Users/hlc/Desktop/UCLA Courses/ECE C147A/Project/stock_return_project4.py
/Users/hlc/Desktop/UCLA Courses/ECE C147A/Project/run_project_mhattn.py


In [2]:
# Quick run defaults for the attention experiment.
START_DATE = "2015-01-01"
END_DATE = "2025-01-01"
LOOKBACK = 60
HORIZON = 5
UNIVERSE = "small"  # small | sp500 | nasdaq100 | auto
MODELS = ["RNN", "LSTM", "GRU", "TRANSFORMER"]
ATTN_HEADS = 4
DEVICE = mh.resolve_device()

UNIVERSE_TO_TICKERS = {
    "small": srp.DEFAULT_LIQUID_TICKERS,
    "sp500": srp.SP500_TICKERS,
    "nasdaq100": srp.NASDAQ100_TICKERS,
    "auto": None,
}
TICKERS = UNIVERSE_TO_TICKERS[UNIVERSE]
print(f"Universe: {UNIVERSE}")

# Universe filter: keep stocks that cover nearly the full sample window.
START_BUFFER_DAYS = 5
END_BUFFER_DAYS = 5
MIN_COVERAGE_RATIO = 0.95


Universe: small


In [3]:
raw_prices = srp.download_price_history(start=START_DATE, end=END_DATE, tickers=TICKERS)

flat_prices = srp.flatten_price_frame(raw_prices).dropna(subset=["Close"]).copy()
all_dates = pd.Index(sorted(pd.to_datetime(flat_prices["Date"].unique())))
date_to_pos = {date: idx for idx, date in enumerate(all_dates)}

coverage = (
    flat_prices.groupby("Ticker")
    .agg(
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        n_dates=("Date", "nunique"),
    )
    .reset_index()
)
coverage["first_pos"] = pd.to_datetime(coverage["first_date"]).map(date_to_pos)
coverage["last_pos"] = pd.to_datetime(coverage["last_date"]).map(date_to_pos)
coverage["coverage_ratio"] = coverage["n_dates"] / len(all_dates)

eligible_tickers = coverage.loc[
    (coverage["first_pos"] <= START_BUFFER_DAYS)
    & (coverage["last_pos"] >= len(all_dates) - 1 - END_BUFFER_DAYS)
    & (coverage["coverage_ratio"] >= MIN_COVERAGE_RATIO),
    "Ticker",
].tolist()

filtered_prices = raw_prices.loc[
    :,
    raw_prices.columns.get_level_values(0).isin(eligible_tickers),
]

print(
    f"Kept {len(eligible_tickers)} / {coverage.shape[0]} tickers "
    f"with >= {MIN_COVERAGE_RATIO:.0%} coverage and near-full sample history."
)
print(
    "Dropped examples:",
    coverage.loc[~coverage["Ticker"].isin(eligible_tickers), "Ticker"].head(10).tolist(),
)

experiment_data = srp.prepare_experiment_data(
    filtered_prices,
    horizon=HORIZON,
    lookback=LOOKBACK,
    train_size=0.7,
    val_size=0.15,
)
for split_name, split_df in experiment_data.splits.items():
    print(split_name, split_df["Date"].min(), split_df["Date"].max(), len(split_df))


Kept 49 / 50 tickers with >= 95% coverage and near-full sample history.
Dropped examples: ['UBER']
train 2015-02-02 00:00:00 2021-12-31 00:00:00 85407
val 2022-01-03 00:00:00 2023-06-29 00:00:00 18326
test 2023-06-30 00:00:00 2024-12-23 00:00:00 18326


In [ ]:
MODEL_CONFIGS = {
    "RNN": srp.TrainConfig(
        batch_size=128,
        hidden_dim=64,
        num_layers=1,
        dropout=0.0,
        learning_rate=1e-4,
        weight_decay=1e-5,
        max_epochs=20,
        patience=6,
        grad_clip=1.0,
        device=DEVICE,
    ),
    "LSTM": srp.TrainConfig(
        batch_size=128,
        hidden_dim=64,
        num_layers=2,
        dropout=0.15,
        learning_rate=3e-4,
        weight_decay=1e-5,
        max_epochs=25,
        patience=6,
        grad_clip=1.0,
        device=DEVICE,
    ),
    "GRU": srp.TrainConfig(
        batch_size=128,
        hidden_dim=64,
        num_layers=2,
        dropout=0.15,
        learning_rate=1e-4,
        weight_decay=1e-5,
        max_epochs=25,
        patience=6,
        grad_clip=1.0,
        device=DEVICE,
    ),
    "TRANSFORMER": srp.TrainConfig(
        batch_size=256,
        hidden_dim=160,
        num_layers=3,
        dropout=0.15,
        num_heads=8,
        learning_rate=3e-4,
        weight_decay=5e-5,
        max_epochs=35,
        patience=10,
        grad_clip=1.0,
        device=DEVICE,
    ),
}

input_dim = len(experiment_data.feature_columns)
results = {}

for model_name in MODELS:
    train_config = MODEL_CONFIGS[model_name]
    print(f"=== Notebook model: {model_name} ===")

    if model_name == "RNN":
        model = srp.build_model(
            model_name="RNN",
            input_dim=input_dim,
            hidden_dim=train_config.hidden_dim,
            num_layers=train_config.num_layers,
            dropout=train_config.dropout,
        )
    elif model_name in {"LSTM", "GRU"}:
        model = mh.AttentionSequenceRegressor(
            input_dim=input_dim,
            hidden_dim=train_config.hidden_dim,
            num_layers=train_config.num_layers,
            dropout=train_config.dropout,
            cell_type=model_name,
            num_attn_heads=ATTN_HEADS,
        )
    elif model_name == "TRANSFORMER":
        model = srp.build_model(
            model_name="TRANSFORMER",
            input_dim=input_dim,
            hidden_dim=train_config.hidden_dim,
            num_layers=train_config.num_layers,
            dropout=train_config.dropout,
            num_heads=train_config.num_heads,
        )
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    results[model_name] = mh.run_experiment(
        model,
        model_name,
        experiment_data,
        train_config,
    )


=== Notebook model: RNN ===

=== Training RNN ===
[train] batch 1/645 - loss: 0.051914
[train] batch 200/645 - loss: 0.005690
[train] batch 400/645 - loss: 0.003913
[train] batch 600/645 - loss: 0.003229
[train] batch 645/645 - loss: 0.003126
[val] batch 1/121 - loss: 0.002845
[val] batch 121/121 - loss: 0.002483
Epoch 1/20 - train_loss: 0.003126, val_loss: 0.002483, time: 10.2s
Validation improved; checkpoint updated.
[train] batch 1/645 - loss: 0.001704
[train] batch 200/645 - loss: 0.001742


In [ ]:
summary = srp.build_summary_frame(results)
summary


# Result Visualizations

Plots for loss curves, IC/RankIC, portfolio performance, and model comparison.


In [ ]:
best_model = summary.iloc[0]["model"]
print("Visualizing model:", best_model)
srp.plot_training_histories(results)


In [ ]:
srp.plot_ic_series(results, best_model, split="test", rolling_window=20)


In [ ]:
srp.plot_portfolio_curve(results, best_model)
srp.plot_summary_bars(summary)


# Prediction diagnostics

Inspect raw predictions, sign accuracy, and the prediction-vs-target scatter plot.


In [ ]:
model_name = best_model
pred = results[model_name]["test_predictions"].copy()
pred.head()


In [ ]:
pred["pred_sign"] = (pred["prediction"] > 0).astype(int)
pred["true_sign"] = (pred["target"] > 0).astype(int)

sign_accuracy = (pred["pred_sign"] == pred["true_sign"]).mean()
daily_sign_accuracy = pred.groupby("Date").apply(
    lambda x: ((x["prediction"] > 0) == (x["target"] > 0)).mean(),
    include_groups=False,
)

print("Overall sign accuracy:", round(float(sign_accuracy), 4))
daily_sign_accuracy.describe()


In [ ]:
import matplotlib.pyplot as plt

latest_day = pred["Date"].max()
latest_slice = pred[pred["Date"] == latest_day].copy()
display(latest_slice.sort_values("prediction", ascending=False).head(10))

ax = latest_slice.plot.scatter(
    x="prediction",
    y="target",
    figsize=(6, 5),
    alpha=0.7,
    title=f"{model_name} prediction vs target on {latest_day.date()}"
)
ax.axhline(0.0, color="black", linewidth=1, alpha=0.6)
ax.axvline(0.0, color="black", linewidth=1, alpha=0.6)
plt.show()
